In [ ]:
import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents,fetch_website_links
from IPython.display import Markdown,display, update_display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()





link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

        
def select_relevant_links(url):
    print(f"selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(model=MODEL, messages=[{"role":"system","content": link_system_prompt}, {"role": "user", "content": get_links_user_prompt(url)}], response_format = {"type":"json_object"})
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"found {len(links["links"])} relevant link")
    return links


def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link["type"]}\n"
        result += fetch_website_contents(link["url"])
    return result



brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt



def create_brochure(company_name, url):
    response = openai.chat.completions.create(model="gpt-5-nano", messages = [{"role": "system", "content":brochure_system_prompt}, {"role":"user", "content": get_brochure_user_prompt(company_name, url)}])
    result = response.choices[0].message.content
    display(Markdown(result))



def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

stream_brochure("learn2earn", "https://learn2earn.ng/")

API key looks good so far
selecting relevant links for https://learn2earn.ng/ by calling gpt-5-nano
found 0 relevant link


# Learn2Earn Brochure

---

## Learn2Earn: Learn to Code, Earn While Learning

### About Us
Learn2Earn is a cutting-edge, AI-focused digital skills training company based in Nigeria. We empower individuals with no prior coding experience to become AI-native full-stack engineers through an immersive, world-class training program. Our mission is to make tech education accessible by eliminating upfront tuition fees, providing real-world experience, and preparing learners for the AI-driven future.

---

### Our Unique Approach

- **Zero Upfront Tuition:** No fees, no student debt. Access top-tier technical training without financial barriers.
- **No Background Required:** Whether you’re new to coding or looking to upskill, our program builds your expertise from the ground up.
- **AI-First Engineering:** Learn not just to code, but to build and innovate with AI and emerging technologies from day one.
- **Paid Work Opportunity:** Gain two years of access to paid work post-training, connecting you directly with global tech companies.
- **On-Campus Experience:** Learn in-person across Nigeria at our state-of-the-art campuses, with a laptop provided for every learner.

---

### Program Highlights

- **Duration:** 2 years total (1 year training + 1 year paid work access)
- **Training Model:** Learn by building real-world projects, collaborate with peers, and utilize today's most in-demand AI and full-stack tools.
- **Global Perspective:** Graduate with skills comparable to a top international CS degree plus professional AI experience.
- **Application Process:** Begin by playing our free 90-minute Selection Game, designed to test memory and problem-solving rather than prior knowledge.

---

### Why Choose Learn2Earn?

- **Build an AI-Native Career:** We train engineers to adapt alongside rapid AI advancements, future-proofing your tech skills.
- **Inclusive & Accessible:** Limited slots available with open applications across Nigeria—bringing high-quality tech education closer to you.
- **Real-World Connection:** Direct pathways to paid roles at global tech companies after graduation ensure your learning translates to career growth.

---

### Our Partners

We collaborate with leading organizations such as Local Globe to expand our impact and provide unparalleled opportunities for our learners.

---

### Careers and Opportunities

Join a vibrant culture of innovation, collaboration, and growth. Whether you’re a prospective learner ready to launch a tech career or a tech professional seeking to contribute as a mentor or staff, Learn2Earn offers meaningful roles in a fast-growing AI-driven ecosystem.

---

### Join Us Today!

- **Location:** Across Nigeria through our on-campus experience.
- **Application:** Play the Selection Game online anytime to qualify for the Trials.
- **Start Your Journey:** Become a future-ready AI-first full-stack engineer with zero upfront costs.

**Website:** [Learn2Earn](#)  
**Apply Now:** Play the Selection Game and seize your spot!

---

**Learn2Earn – From Potential to AI Engineer.**  
Learn. Build. Earn. Thrive.